# SAM2 Hybrid — Encode an image, ship a decoder bundle

Companion notebook for [Making AI feel realtime with hybrid segmentation](https://jeanrojas.com/blog/splitting-sam2-encoder-decoder).

**What this notebook does**
1. Installs SAM2 + the ONNX toolchain.
2. Downloads `sam2.1_hiera_large.pt` (~900 MB) from Meta.
3. Splits it into encoder + decoder ONNX files via `samexporter`.
4. Encodes any image you upload and produces a `~16 MB` `embedding.bin` + `manifest.json` bundle that the browser-side decoder consumes directly.

**Where to run it**
- **Colab T4** (Runtime → Change runtime type → T4 GPU). Total: ~2 minutes after the checkpoint download.
- A local Jupyter / VS Code kernel with **Python ≥ 3.11**. CUDA, CoreML, or CPU all work — `onnxruntime` picks the fastest provider available.

> ⚠️ **Run the cells top-to-bottom.** Cell 1 installs everything; running `samexporter` from a terminal without first running Cell 1 will fail with `No module named 'samexporter'`.

## Cell 1 — Install dependencies

We let `samexporter` pin its own `torch` / `onnxruntime` / `onnx` versions —
it has hard pins (e.g. `torch==2.10.0`) that break if we set them ourselves.
Then we add the SAM2 codebase from Meta on top.

In [ ]:
import sys
assert sys.version_info >= (3, 11), (
    f"samexporter requires Python ≥ 3.11; you have {sys.version}. "
    "On Colab: Runtime → Change runtime type → Python 3.11+. "
    "Locally: pyenv install 3.11 / use a 3.11 conda env."
)

# Install samexporter and its hard-pinned deps. NO torch pin from us.
%pip install --upgrade samexporter

# Add Meta's SAM2 model code (separate from the ONNX exporter).
%pip install --upgrade "git+https://github.com/facebookresearch/segment-anything-2.git"

## Cell 1b — Verify the install

If anything failed silently above, this cell surfaces it loudly before we
spend 90 seconds downloading a 900 MB checkpoint.

In [ ]:
import importlib, importlib.util

missing = []
for mod in ["samexporter", "torch", "onnx", "onnxruntime", "PIL", "numpy"]:
    if importlib.util.find_spec(mod) is None:
        missing.append(mod)

if missing:
    raise RuntimeError(
        f"Missing modules after install: {missing}. "
        f"Re-run Cell 1; if it still fails, check the pip output for resolution errors."
    )

import torch, onnxruntime as ort, samexporter
print(f"samexporter   : {samexporter.__file__}")
print(f"torch         : {torch.__version__}  (cuda: {torch.cuda.is_available()})")
print(f"onnxruntime   : {ort.__version__}")
print(f"ORT providers : {ort.get_available_providers()}")

## Cell 2 — Download the SAM2.1 Hiera Large checkpoint (~900 MB)

In [ ]:
import urllib.request
from pathlib import Path

CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt"
ckpt_path = Path("original_models/sam2.1_hiera_large.pt")
ckpt_path.parent.mkdir(exist_ok=True)

if not ckpt_path.exists():
    print("Downloading sam2.1_hiera_large.pt (~900 MB)…")
    urllib.request.urlretrieve(CHECKPOINT_URL, ckpt_path)
print(f"Checkpoint at {ckpt_path}  ({ckpt_path.stat().st_size / 1e6:.1f} MB)")

## Cell 3 — Split into encoder.onnx + decoder.onnx via `samexporter`

The encoder for the large variant exceeds the 2 GB ONNX limit, so the
exporter automatically emits an `.onnx` file plus an external `.onnx.data`
file containing the weights. Both must travel together at load time.

We invoke samexporter through `sys.executable -m` so the CLI is guaranteed
to use **the same Python interpreter** that just installed the package — a
common gotcha when running inside a notebook with multiple Python envs.

In [ ]:
import subprocess, sys

Path("output_models").mkdir(exist_ok=True)

result = subprocess.run(
    [
        sys.executable, "-m", "samexporter.export_sam2",
        "--checkpoint",       "original_models/sam2.1_hiera_large.pt",
        "--output_encoder",   "output_models/sam2.1_hiera_large.encoder.onnx",
        "--output_decoder",   "output_models/sam2.1_hiera_large.decoder.onnx",
        "--model_type",       "sam2.1_hiera_large",
    ],
    check=False,
)
if result.returncode != 0:
    raise RuntimeError(
        f"samexporter exited with code {result.returncode}. "
        "Most common cause: torch/onnx version mismatch. "
        "Re-run Cell 1 (do NOT re-pin torch yourself) and try again."
    )

for p in sorted(Path("output_models").glob("*")):
    print(f"  {p}  ({p.stat().st_size / 1e6:.1f} MB)")

## Cell 4 — Encode an image

Replace `IMAGE_PATH` with whatever you uploaded. On Colab use the file
browser on the left to drop a JPG/PNG into the working directory.

In [ ]:
import numpy as np
import onnxruntime as ort
from PIL import Image

IMAGE_PATH = "my_photo.jpg"

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
INPUT_SIZE = 1024

def preprocess(image_path):
    img = Image.open(image_path).convert("RGB")
    original_size = img.size  # (width, height)
    img = img.resize((INPUT_SIZE, INPUT_SIZE), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = (arr - MEAN) / STD
    arr = arr.transpose(2, 0, 1)[None]  # (1, 3, 1024, 1024)
    return arr.astype(np.float32), original_size

encoder = ort.InferenceSession(
    "output_models/sam2.1_hiera_large.encoder.onnx",
    providers=ort.get_available_providers(),
)

input_tensor, original_size = preprocess(IMAGE_PATH)
high_res_0, high_res_1, image_embed = encoder.run(None, {"image": input_tensor})

print("image_embed:     ", image_embed.shape, image_embed.dtype)
print("high_res_feats_0:", high_res_0.shape)
print("high_res_feats_1:", high_res_1.shape)

## Cell 5 — Pack as a browser-friendly bundle

Two compression tricks: cast to `float16` (halves size, decoder is robust
to it) and lay it out as a flat binary blob with a JSON manifest.
Combined the bundle lands at ~12–18 MB depending on the image.

In [ ]:
import json

tensors = {
    "image_embed":      image_embed.astype(np.float16),
    "high_res_feats_0": high_res_0.astype(np.float16),
    "high_res_feats_1": high_res_1.astype(np.float16),
}

manifest = {
    "preview": "preview.jpg",
    "originalWidth":  original_size[0],
    "originalHeight": original_size[1],
    "tensors": {},
}

with open("embedding.bin", "wb") as f:
    offset = 0
    for name, arr in tensors.items():
        manifest["tensors"][name] = {
            "offset": offset,
            "shape":  list(arr.shape),
            "dtype":  "float16",
        }
        f.write(arr.tobytes())
        offset += arr.nbytes
    manifest["totalBytes"] = offset

with open("manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

preview = Image.open(IMAGE_PATH).convert("RGB")
preview.thumbnail((1600, 1600))
preview.save("preview.jpg", quality=85)

print("Done. Upload these three files to the web app:")
print("  - embedding.bin  ({:.1f} MB)".format(Path("embedding.bin").stat().st_size / 1e6))
print("  - manifest.json")
print("  - preview.jpg    ({:.1f} KB)".format(Path("preview.jpg").stat().st_size / 1e3))